# Egyptian ID pipeline (local)

This follows the same **stages** as [`egyptian_id_ocr.ipynb`](egyptian_id_ocr.ipynb):

1. **Model 1** — Field detection (`Egyptian-ID-Detectr-3` / `train_id_detectr_hyper`)
2. **Model 2** — Arabic digit boxes (`arabic-numbers` / `train_arabic_numbers_v2`)
3. **Model 3** — National ID region on card (`National-ID-7` / `train_national_id_v7`)
4. **OCR** — 14-digit NID from digit crops; name & address via fields + Tesseract/EasyOCR

**Differences from the Colab notebook:** local `DATASET_ROOT`, optional GPU (no hard `assert cuda`), Windows-friendly commands, **no API keys in the notebook** (use `ROBOFLOW_API_KEY` in the environment if you re-download datasets).


In [ ]:
from pathlib import Path
import os
import sys

# Repository root (this notebook lives here)
DATASET_ROOT = Path(r"C:\Users\yassi\Downloads\dataset").resolve()
assert DATASET_ROOT.is_dir(), f"Set DATASET_ROOT to your clone: {DATASET_ROOT}"
os.chdir(DATASET_ROOT)
if str(DATASET_ROOT) not in sys.path:
    sys.path.insert(0, str(DATASET_ROOT))

DATA_YAML_FIELDS = DATASET_ROOT / "egyptian_id_detectr" / "content" / "Egyptian-ID-Detectr-3" / "data.yaml"
DATA_YAML_DIGITS = DATASET_ROOT / "arabic_numbers" / "content" / "arabic-numbers-2" / "data.yaml"
DATA_YAML_NID = DATASET_ROOT / "national_id" / "content" / "National-ID-7" / "data.yaml"

RUNS = DATASET_ROOT / "runs"
WEIGHTS_FIELDS = RUNS / "train_id_detectr_hyper" / "weights" / "best.pt"
WEIGHTS_DIGITS = RUNS / "train_arabic_numbers_v2" / "weights" / "best.pt"
WEIGHTS_NID_CARD = RUNS / "train_national_id_v7" / "weights" / "best.pt"

for label, p in [
    ("fields data.yaml", DATA_YAML_FIELDS),
    ("digits data.yaml", DATA_YAML_DIGITS),
    ("NID card data.yaml", DATA_YAML_NID),
]:
    print(label + ":", "OK" if p.is_file() else "MISSING", p)

for label, p in [
    ("field detector", WEIGHTS_FIELDS),
    ("Arabic digit YOLO", WEIGHTS_DIGITS),
    ("national-ID-on-card YOLO", WEIGHTS_NID_CARD),
]:
    print(label + ":", "OK" if p.is_file() else "MISSING (train first)", p)


In [ ]:
# Optional GPU check (reference notebook used a hard CUDA assert; here it is optional)
import torch
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| torch:", torch.__version__)
else:
    print("CUDA not available — YOLO/OCR will use CPU (slower). torch:", torch.__version__)


## Model 1 — Train field detector (optional)

Reference: `egyptian_id_ocr.ipynb` cells with `name=train_id_detectr_v1` / `train_id_detectr_hyper`.

On **Windows**, use `workers=0` in Python API or add `workers=0` to CLI to avoid dataloader issues.


In [ ]:
# Example: train field model (uncomment to run; long-running)
# from ultralytics import YOLO
# model = YOLO("yolov8n.pt")
# model.train(data=str(DATA_YAML_FIELDS), epochs=30, imgsz=640, batch=16, device=0, workers=0,
#             project=str(RUNS), name="train_id_detectr_hyper")


## Model 2 — Arabic digit detector (optional)

Reference: `arabic-numbers` Roboflow + `train_arabic_numbers_v2`.


In [ ]:
# from ultralytics import YOLO
# model = YOLO("yolov8s.pt")
# model.train(data=str(DATA_YAML_DIGITS), epochs=20, imgsz=640, batch=16, device=0, workers=0,
#             project=str(RUNS), name="train_arabic_numbers_v2")


## Model 3 — National ID region on card (optional)

Reference: `national-id` Roboflow + `train_national_id_v7`.


In [ ]:
# from ultralytics import YOLO
# model = YOLO("yolov8n.pt")
# model.train(data=str(DATA_YAML_NID), epochs=40, imgsz=640, batch=16, device=0, workers=0,
#             project=str(RUNS), name="train_national_id_v7")


## Re-download datasets via Roboflow (optional)

The reference notebook used `Roboflow(api_key="...")`. **Do not paste keys into notebooks.** Use:

```python
import os
from roboflow import Roboflow
rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
```


In [ ]:
# Example only — requires ROBOFLOW_API_KEY in the environment
# from roboflow import Roboflow
# import os
# rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
# # project = rf.workspace("...").project("...")
# # project.version(3).download("yolov8")


## End-to-end on one image (matches notebook intent, uses your scripts)

- **Fields + Excel export (Tesseract):** `export_id_to_excel.py`
- **14-digit NID from digit YOLO:** `extract_nid_digits.py` with `--nid-field-weights` + `--reading-order ltr`
- **Name + address (EasyOCR recommended for screenshots):** `extract_name_address.py` with default `--engine mixed`

Set `IMAGE` to a quoted path if it contains spaces.


In [ ]:
IMAGE = DATASET_ROOT / "egyptian_id_detectr" / "content" / "Egyptian-ID-Detectr-3" / "train" / "images"
# Pick one jpg from train/images for a smoke test, or set a fixed file:
candidates = sorted(IMAGE.glob("*.jpg")) if IMAGE.is_dir() else []
SAMPLE = candidates[0] if candidates else None
print("Sample image:", SAMPLE)


In [ ]:
import subprocess

def run_py(script: str, args: list[str]) -> None:
    cmd = [sys.executable, str(DATASET_ROOT / script)] + args
    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd=DATASET_ROOT, check=True)

# if SAMPLE is not None:
#     run_py("extract_nid_digits.py", [str(SAMPLE), "--nid-field-weights", str(WEIGHTS_FIELDS), "--reading-order", "ltr"])
#     run_py("extract_name_address.py", [str(SAMPLE), "--engine", "mixed", "--save-crops", str(RUNS / "id_export" / "nb_crops")])
#     run_py("export_id_to_excel.py", [str(SAMPLE), "--output", str(RUNS / "id_export" / "nb_export.xlsx")])


## In-notebook inference (same building blocks as old `extract_text` cells)

The reference notebook cropped YOLO boxes and ran `pytesseract` with a simple threshold. Below: **YOLO predict → crop →** reuse `export_id_to_excel` helpers for consistency with `extract_name_address.py`.


In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
import export_id_to_excel as eid

if not WEIGHTS_FIELDS.is_file() or SAMPLE is None:
    print("Skip: need field weights and SAMPLE image")
else:
    bgr = cv2.imread(str(SAMPLE))
    model = YOLO(str(WEIGHTS_FIELDS))
    dev = 0 if torch.cuda.is_available() else "cpu"
    r = model.predict(bgr, conf=0.25, device=dev, verbose=False)[0]
    names = eid.load_class_names()
    xyxy = r.boxes.xyxy.cpu().numpy()
    cls = r.boxes.cls.cpu().numpy().astype(int)
    confs = r.boxes.conf.cpu().numpy()
    best = eid.best_boxes_by_label(xyxy, cls, confs, names)
    for lab in ("firstName", "lastName", "address", "nid"):
        if lab not in best:
            print(lab, ": (no box)")
            continue
        crop = eid.crop_xyxy(bgr, best[lab][0], pad=6)
        t = eid.ocr_crop(crop, lang="ara+eng", psm=6)
        print(lab, ":", repr(t[:120]))
